# 06 — Final CNN Model: EfficientNetB0 V3

> **Notebook status: reconstructed training record, not the original training notebook.** The packaged model, inference configuration and project run notes preserve the architecture and selected training settings. The exact `train_improved.py` source script and original training history/epoch logs were not present in the uploaded application ZIP or the recovered project files. This notebook explains the documented run and shows architecture/training code structure; it does not claim to be the original executed training notebook.

## Model identity

The final application uses one model, shown to ordinary users as **CNN**. Its technical backbone is ImageNet-pretrained **EfficientNetB0** fine-tuned for binary classification.

- Input: 256 × 256 × 3 RGB.
- Labels: `0 = real`, `1 = ai_generated`.
- Output: one sigmoid score.
- Threshold used by the app: 0.47.
- Saved file: `models_v3/car_cnn_model_v3.keras`.

## Recorded architecture

1. Input layer (256 × 256 × 3, float32 RGB values 0–255).
2. Training augmentation: horizontal flip, rotation factor 0.04, zoom factor 0.10, contrast factor 0.10 and brightness factor 0.10.
3. EfficientNetB0 pretrained backbone with `include_top=False` and built-in input rescaling/normalisation.
4. GlobalAveragePooling2D.
5. Dropout(0.40).
6. Dense(64, ReLU) with L2 regularisation 1e-4.
7. Dropout(0.30).
8. Dense(1, sigmoid).

The saved model's serialized configuration confirms the 256×256×3 input, augmentation layers, EfficientNetB0 backbone and embedded normalization. Inference supplies raw 0–255 float32 RGB values; do not divide by 255 externally a second time.


In [ ]:
from pathlib import Path
import json, hashlib, zipfile

PROJECT_ROOT = Path.cwd()
CONFIG_PATH = PROJECT_ROOT / "config" / "cnn_inference_config.json"
MODEL_PATH = PROJECT_ROOT / "models_v3" / "car_cnn_model_v3.keras"
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

print("Model display name:", config["model_name"])
print("Technical architecture:", config["architecture"]["type"])
print("Input:", config["architecture"]["input_shape"])
print("Threshold:", config["decision_threshold"])
print("Model present:", MODEL_PATH.is_file())
if MODEL_PATH.is_file():
    digest = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
    print("SHA-256:", digest)
    assert digest == config["model_sha256"]
    with zipfile.ZipFile(MODEL_PATH) as saved:
        model_config = json.loads(saved.read("config.json"))
    print("Serialized model class:", model_config.get("class_name"))
    print("Serialized config keys:", list(model_config.get("config", {}).keys()))
else:
    print("Run this cell from the repository root to inspect the preserved model.")


## Training data coverage

| Split | Expected candidates | Loaded by the V3 Keras directory loader | Skipped WebP files |
|---|---:|---:|---:|
| Training | 874 | 872 | 2 |
| Validation | 188 | 185 | 3 |
| Testing | 188 | 186 | 2 |

The V3 metrics use these smaller effective counts. The full split remains 1,250 candidates, but the model run did not load all of them. This limitation must remain visible in any explanation or report.


## Two-phase training procedure

### Phase 1 — Frozen backbone

- Use ImageNet-pretrained EfficientNetB0.
- Freeze the backbone and train the classification head only.
- Optimizer: Adam.
- Learning rate: 0.001.
- Maximum epochs: 15.

### Phase 2 — Fine-tuning

- Unfreeze the top 40 backbone layers.
- Keep all Batch Normalization layers frozen.
- Optimizer: Adam.
- Learning rate: 0.00001.
- Maximum epochs: 25.

Other recorded techniques: binary cross-entropy with label smoothing 0.05, data augmentation, dropout, L2 regularisation, early stopping and learning-rate reduction. Exact callback patience/factor settings and batch size were not found in the recovered files, so they are not guessed here.

The model configuration records training seed 42 and dataset split seed 211. The test split was not used during training or threshold tuning.


In [ ]:
# Optional architecture reconstruction for study only. This cell is not run automatically.
# It may download ImageNet weights if RUN_ARCHITECTURE_BUILD is changed to True.
RUN_ARCHITECTURE_BUILD = False
if RUN_ARCHITECTURE_BUILD:
    import tensorflow as tf
    from tensorflow.keras import layers, regularizers, Model
    inputs = tf.keras.Input(shape=(256, 256, 3))
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.04)(x)
    x = layers.RandomZoom(0.10)(x)
    x = layers.RandomContrast(0.10)(x)
    x = layers.RandomBrightness(0.10, value_range=(0, 255))(x)
    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet", input_shape=(256, 256, 3)
    )
    backbone.trainable = False
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.40)(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    reconstructed_model = Model(inputs, outputs, name="car_authenticity_effnet")
    reconstructed_model.summary()
else:
    print("Architecture build is disabled. This notebook does not download weights or train a model by default.")


## Training workflow (settings known; callback details incomplete)

The following is a run-plan, not a ready-to-execute retraining command. It is shown so a reviewer can follow the two training phases. To reproduce the original run exactly, the missing original training script, callback values, batch size, dataset loader arguments and training logs are still required. No retraining is performed as part of this package revision.

```python
# Phase 1
backbone.trainable = False
compile(model, optimizer=Adam(learning_rate=0.001),
        loss=BinaryCrossentropy(label_smoothing=0.05))
fit(model, training_data, validation_data=validation_data, epochs=15,
    callbacks=[early_stopping, reduce_learning_rate])

# Phase 2
backbone.trainable = True
for layer in backbone.layers[:-40]:
    layer.trainable = False
for layer in backbone.layers:
    if isinstance(layer, BatchNormalization):
        layer.trainable = False
compile(model, optimizer=Adam(learning_rate=0.00001),
        loss=BinaryCrossentropy(label_smoothing=0.05))
fit(model, training_data, validation_data=validation_data, epochs=25,
    callbacks=[early_stopping, reduce_learning_rate])
```

The `callbacks`, exact batch size and data-loading arguments above are intentionally not instantiated because their historical values are not present in the recovered source. This avoids silently creating a different experiment and claiming it is the original run.